# Etapa 12 — desempenho computacional

Este notebook completa a comparação do experimento sintético medindo a latência das implementações espacial, Forward e Viterbi. Ele reutiliza as funções de `01_cenario_sintetico.ipynb` e os parâmetros congelados em `manifesto_teste_final.json`.

## Delimitação

- As mesmas quatro rotas, seis fatores de ruído e 1.000 sementes do teste final são usados: 24.000 sequências e 144.000 observações.
- A geração das observações, a construção da matriz de transição, a leitura de arquivos e a gravação dos resultados ficam fora do cronômetro.
- Cada chamada processa uma sequência completa de seis observações.
- A ordem dos métodos é aleatorizada de forma reprodutível para reduzir viés de ordem.
- Há aquecimento antes das medições.
- São reportadas a mediana e o percentil 95 (P95), além da média e do intervalo interquartil.
- Os números caracterizam esta implementação Python no computador informado; não medem o desempenho no headset ou em uma implantação offshore.

## Pipelines medidos

1. `espacial`: distância de Mahalanobis e escolha do candidato mais próximo, sem abstenção.
2. `forward`: probabilidades de emissão, recursão Forward e escolha do candidato mais provável, sem abstenção.
3. `viterbi`: probabilidades de emissão e decodificação Viterbi da sequência.
4. `espacial_com_abstencao`: probabilidades de emissão e regra de suspensão com os limiares finais.
5. `forward_com_abstencao`: probabilidades de emissão, recursão Forward e regra de suspensão com os limiares finais.


In [1]:
from __future__ import annotations

import ast
import contextlib
import gc
import hashlib
import io
import json
import os
import platform
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import nbformat
import networkx as nx
import numpy as np
import pandas as pd


def localizar_raiz_repositorio():
    candidatos = [Path.cwd(), *Path.cwd().parents]

    for candidato in candidatos:
        notebook_base = (
            candidato
            / "notebooks"
            / "01_cenario_sintetico.ipynb"
        )

        if notebook_base.is_file():
            return candidato.resolve()

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do repositório."
    )


RAIZ = localizar_raiz_repositorio()
NOTEBOOK_BASE = (
    RAIZ / "notebooks" / "01_cenario_sintetico.ipynb"
)
PASTA_RESULTADOS = RAIZ / "resultados"
MANIFESTO_TESTE = (
    PASTA_RESULTADOS / "manifesto_teste_final.json"
)

print("Raiz:", RAIZ)
print("Notebook-base:", NOTEBOOK_BASE.relative_to(RAIZ))
print("Manifesto:", MANIFESTO_TESTE.relative_to(RAIZ))


Raiz: /home/schwdtr/Projetos/XRlab/GEORREF. E RED.VISUAL/REDUNDÂNCIA VISUAL/Implementação/HMM_ASSET_DESAMBIGUATION
Notebook-base: notebooks/01_cenario_sintetico.ipynb
Manifesto: resultados/manifesto_teste_final.json


## 1. Carregamento controlado da implementação

As células preparatórias do notebook original são executadas somente até a definição da regra de abstenção. Na célula dessa regra, apenas a função é carregada; os experimentos de validação e teste não são repetidos durante a importação.


In [2]:
def carregar_implementacao_base(caminho_notebook):
    notebook = nbformat.read(
        caminho_notebook,
        as_version=4,
    )
    namespace = {
        "__name__": "benchmark_notebook_base",
        "display": lambda *args, **kwargs: None,
    }

    celula_limite = None
    mostrar_figuras_original = plt.show

    try:
        with contextlib.redirect_stdout(
            io.StringIO()
        ), contextlib.redirect_stderr(io.StringIO()):
            for numero, celula in enumerate(
                notebook.cells,
                start=1,
            ):
                if celula.cell_type != "code":
                    continue

                arvore = ast.parse(celula.source)
                definicoes_abstencao = [
                    item
                    for item in arvore.body
                    if isinstance(item, ast.FunctionDef)
                    and item.name == "aplicar_regra_abstencao"
                ]

                if definicoes_abstencao:
                    modulo = ast.Module(
                        body=definicoes_abstencao,
                        type_ignores=[],
                    )
                    ast.fix_missing_locations(modulo)
                    exec(
                        compile(
                            modulo,
                            filename=(
                                f"{caminho_notebook}:"
                                f"celula-{numero}"
                            ),
                            mode="exec",
                        ),
                        namespace,
                    )
                    celula_limite = numero
                    break

                exec(
                    compile(
                        celula.source,
                        filename=(
                            f"{caminho_notebook}:"
                            f"celula-{numero}"
                        ),
                        mode="exec",
                    ),
                    namespace,
                )

                if "plt" in namespace:
                    namespace["plt"].show = (
                        lambda *args, **kwargs: None
                    )
                    namespace["plt"].close("all")
    finally:
        plt.show = mostrar_figuras_original
        plt.close("all")

    if celula_limite is None:
        raise RuntimeError(
            "A função aplicar_regra_abstencao "
            "não foi encontrada."
        )

    return namespace, celula_limite


base, celula_limite = carregar_implementacao_base(
    NOTEBOOK_BASE
)

nomes_obrigatorios = [
    "ativos",
    "G_mov",
    "maximo_arestas",
    "sequencias_inspecao",
    "desvios_padrao",
    "identidades_ativos",
    "distribuicao_inicial",
    "simular_observacoes",
    "associar_independentemente",
    "construir_matriz_transicao",
    "construir_matriz_emissoes",
    "executar_forward",
    "executar_viterbi",
    "aplicar_regra_abstencao",
]

ausentes = [
    nome
    for nome in nomes_obrigatorios
    if nome not in base
]

if ausentes:
    raise RuntimeError(
        f"Objetos ausentes no notebook-base: {ausentes}"
    )

print("Implementação carregada até a célula:", celula_limite)
print("Objetos obrigatórios presentes:", not ausentes)


Implementação carregada até a célula: 36
Objetos obrigatórios presentes: True


## 2. Configuração final e ambiente

O manifesto publicado é a fonte dos fatores experimentais e dos limiares. A matriz de transição é reconstruída com a escala final de 2,0 m.


In [3]:
manifesto_teste = json.loads(
    MANIFESTO_TESTE.read_text(encoding="utf-8")
)

ativos = base["ativos"]
grafo_movimentacao = base["G_mov"]
maximo_arestas = base["maximo_arestas"]
sequencias_inspecao = base["sequencias_inspecao"]
desvios_padrao = np.asarray(
    base["desvios_padrao"],
    dtype=float,
)
identidades_ativos = list(base["identidades_ativos"])
distribuicao_inicial = (
    base["distribuicao_inicial"].copy()
)

simular_observacoes = base["simular_observacoes"]
associar_independentemente = (
    base["associar_independentemente"]
)
construir_matriz_transicao = (
    base["construir_matriz_transicao"]
)
construir_matriz_emissoes = (
    base["construir_matriz_emissoes"]
)
executar_forward = base["executar_forward"]
executar_viterbi = base["executar_viterbi"]
aplicar_regra_abstencao = (
    base["aplicar_regra_abstencao"]
)

escala_transicao = float(
    manifesto_teste["escala_transicao_m"]
)
fatores_ruido = [
    float(valor)
    for valor in manifesto_teste["fatores_ruido"]
]
rotas_teste = [
    *manifesto_teste["rotas_primarias"],
    *manifesto_teste["rotas_estresse"],
]
sementes_teste = range(
    int(manifesto_teste["sementes"]["inicio"]),
    int(manifesto_teste["sementes"]["fim_inclusivo"])
    + 1,
)
limiares_espacial = (
    manifesto_teste["limiares"]["espacial"]
)
limiares_forward = (
    manifesto_teste["limiares"]["forward"]
)

matriz_transicao = construir_matriz_transicao(
    grafo=grafo_movimentacao,
    identidades=identidades_ativos,
    escala=escala_transicao,
    maximo_arestas=maximo_arestas,
)

assert np.isclose(escala_transicao, 2.0)
assert len(rotas_teste) == 4
assert len(fatores_ruido) == 6
assert len(sementes_teste) == 1000
assert np.allclose(matriz_transicao.sum(axis=1), 1.0)


def obter_modelo_processador():
    caminho = Path("/proc/cpuinfo")

    if caminho.is_file():
        for linha in caminho.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines():
            if linha.lower().startswith("model name"):
                return linha.split(":", 1)[1].strip()

    return platform.processor() or "não informado"


def hash_fontes_notebook(caminho):
    notebook = nbformat.read(caminho, as_version=4)
    fontes = [
        {
            "tipo": celula.cell_type,
            "fonte": celula.source,
        }
        for celula in notebook.cells
    ]
    conteudo = json.dumps(
        fontes,
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.sha256(conteudo).hexdigest()


commit_base = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=RAIZ,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

ambiente = {
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
    "plataforma": platform.platform(),
    "processador": obter_modelo_processador(),
    "nucleos_logicos": os.cpu_count(),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "networkx": nx.__version__,
    "commit_base": commit_base,
    "sha256_fontes_notebook_base": (
        hash_fontes_notebook(NOTEBOOK_BASE)
    ),
}

print("Rotas:", rotas_teste)
print("Fatores de ruído:", fatores_ruido)
print(
    "Sementes:",
    min(sementes_teste),
    "a",
    max(sementes_teste),
)
print("Escala de transição:", escala_transicao)
display(pd.Series(ambiente, name="valor").to_frame())


Rotas: ['circuito_horario', 'circuito_antihorario', 'troca_central_longa', 'inspecao_com_saltos']
Fatores de ruído: [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
Sementes: 10000 a 10999
Escala de transição: 2.0


,valor
gerado_em_utc,2026-09-07T03:48:40.980155+00:00
plataforma,Linux-6.12.107+deb13-amd64-x86_64-with-glibc2.41
processador,13th Gen Intel(R) Core(TM) i7-13620H
nucleos_logicos,16
python,3.13.5
numpy,2.5.2
pandas,3.0.5
networkx,3.6.1
commit_base,099ba99da5954a755afbe6b53971265a0ba46f3a
sha256_fontes_notebook_base,49c2cc56777768f7078d1c070bb9ad221626a4e9bda262...


## 3. Implementações cronometradas

As funções abaixo apenas agrupam os componentes já validados. A simulação recebe o mesmo ruído usado no teste final: primeiro o fator multiplica os desvios-padrão; depois a matriz de covariância é construída elevando-os ao quadrado.


In [4]:
def pipeline_espacial(
    observacoes,
    covariancia,
):
    return associar_independentemente(
        observacoes=observacoes,
        ativos=ativos,
        covariancia=covariancia,
    )["predicao_espacial"]


def calcular_emissoes(
    observacoes,
    covariancia,
):
    return construir_matriz_emissoes(
        observacoes=observacoes,
        ativos=ativos,
        covariancia=covariancia,
        identidades=identidades_ativos,
    )


def pipeline_forward(
    observacoes,
    covariancia,
):
    emissoes = calcular_emissoes(
        observacoes,
        covariancia,
    )
    posteriores = executar_forward(
        matriz_emissoes=emissoes,
        matriz_transicao=matriz_transicao,
        distribuicao_inicial=distribuicao_inicial,
    )
    return posteriores.idxmax(axis=1)


def pipeline_viterbi(
    observacoes,
    covariancia,
):
    emissoes = calcular_emissoes(
        observacoes,
        covariancia,
    )
    return executar_viterbi(
        matriz_emissoes=emissoes,
        matriz_transicao=matriz_transicao,
        distribuicao_inicial=distribuicao_inicial,
    )


def pipeline_espacial_com_abstencao(
    observacoes,
    covariancia,
):
    emissoes = calcular_emissoes(
        observacoes,
        covariancia,
    )
    return aplicar_regra_abstencao(
        matriz_probabilidades=emissoes,
        limiar_confianca=float(
            limiares_espacial["limiar_confianca"]
        ),
        limiar_separacao=float(
            limiares_espacial["limiar_separacao"]
        ),
    )


def pipeline_forward_com_abstencao(
    observacoes,
    covariancia,
):
    emissoes = calcular_emissoes(
        observacoes,
        covariancia,
    )
    posteriores = executar_forward(
        matriz_emissoes=emissoes,
        matriz_transicao=matriz_transicao,
        distribuicao_inicial=distribuicao_inicial,
    )
    return aplicar_regra_abstencao(
        matriz_probabilidades=posteriores,
        limiar_confianca=float(
            limiares_forward["limiar_confianca"]
        ),
        limiar_separacao=float(
            limiares_forward["limiar_separacao"]
        ),
    )


pipelines = {
    "espacial": pipeline_espacial,
    "forward": pipeline_forward,
    "viterbi": pipeline_viterbi,
    "espacial_com_abstencao": (
        pipeline_espacial_com_abstencao
    ),
    "forward_com_abstencao": (
        pipeline_forward_com_abstencao
    ),
}

print("Pipelines:", list(pipelines))


Pipelines: ['espacial', 'forward', 'viterbi', 'espacial_com_abstencao', 'forward_com_abstencao']


## 4. Validação e aquecimento

Antes do benchmark, verificamos que a decisão espacial por distância coincide com o máximo das emissões e que todos os pipelines devolvem seis decisões.


In [5]:
nome_rota_aquecimento = rotas_teste[0]
sequencia_aquecimento = sequencias_inspecao[
    nome_rota_aquecimento
]
fator_aquecimento = fatores_ruido[0]
covariancia_aquecimento = np.diag(
    (desvios_padrao * fator_aquecimento) ** 2
)
observacoes_aquecimento = simular_observacoes(
    sequencia=sequencia_aquecimento,
    covariancia=covariancia_aquecimento,
    semente=min(sementes_teste),
)

predicao_distancia = pipeline_espacial(
    observacoes_aquecimento,
    covariancia_aquecimento,
).tolist()
predicao_emissoes = calcular_emissoes(
    observacoes_aquecimento,
    covariancia_aquecimento,
).idxmax(axis=1).tolist()

assert predicao_distancia == predicao_emissoes

for nome_metodo, funcao in pipelines.items():
    resultado = funcao(
        observacoes_aquecimento,
        covariancia_aquecimento,
    )
    assert len(resultado) == len(sequencia_aquecimento)

for _ in range(10):
    for funcao in pipelines.values():
        resultado = funcao(
            observacoes_aquecimento,
            covariancia_aquecimento,
        )
        del resultado

print("Equivalência espacial confirmada: True")
print("Dimensões dos pipelines confirmadas: True")
print("Aquecimento concluído: True")


Equivalência espacial confirmada: True
Dimensões dos pipelines confirmadas: True
Aquecimento concluído: True


## 5. Benchmark nas 24 condições finais

Esta é a célula demorada. Ela realiza 120.000 medições: cinco pipelines para cada uma das 24.000 sequências. O progresso é informado ao final de cada combinação de rota e ruído.


In [6]:
gerador_ordem = random.Random(20260908)
registros = []

quantidade_condicoes = (
    len(rotas_teste) * len(fatores_ruido)
)
numero_condicao = 0
inicio_benchmark = time.perf_counter()

gc.collect()
gc_estava_ativo = gc.isenabled()
gc.disable()

try:
    for nome_rota in rotas_teste:
        sequencia = sequencias_inspecao[nome_rota]

        for fator_ruido in fatores_ruido:
            numero_condicao += 1
            covariancia = np.diag(
                (desvios_padrao * fator_ruido) ** 2
            )

            for semente in sementes_teste:
                observacoes = simular_observacoes(
                    sequencia=sequencia,
                    covariancia=covariancia,
                    semente=semente,
                )

                ordem_metodos = list(pipelines)
                gerador_ordem.shuffle(ordem_metodos)

                for nome_metodo in ordem_metodos:
                    inicio = time.perf_counter_ns()
                    resultado = pipelines[nome_metodo](
                        observacoes,
                        covariancia,
                    )
                    fim = time.perf_counter_ns()

                    registros.append(
                        {
                            "metodo": nome_metodo,
                            "nome_sequencia": nome_rota,
                            "fator_ruido": fator_ruido,
                            "semente": semente,
                            "quantidade_observacoes": (
                                len(sequencia)
                            ),
                            "tempo_ns": fim - inicio,
                        }
                    )
                    del resultado

            decorrido = time.perf_counter() - inicio_benchmark
            print(
                f"[{numero_condicao:02d}/"
                f"{quantidade_condicoes:02d}] "
                f"{nome_rota}, ruído {fator_ruido:.2f}: "
                f"concluído em {decorrido:.1f} s",
                flush=True,
            )
finally:
    if gc_estava_ativo:
        gc.enable()

tempo_total_benchmark_s = (
    time.perf_counter() - inicio_benchmark
)
latencias = pd.DataFrame(registros)

quantidade_sequencias = (
    len(rotas_teste)
    * len(fatores_ruido)
    * len(sementes_teste)
)
quantidade_medicoes = (
    quantidade_sequencias * len(pipelines)
)

assert len(latencias) == quantidade_medicoes
assert (latencias["tempo_ns"] > 0).all()
assert latencias.groupby("metodo").size().eq(
    quantidade_sequencias
).all()

print()
print("Benchmark concluído.")
print("Sequências:", quantidade_sequencias)
print("Medições:", len(latencias))
print(
    "Tempo total:",
    f"{tempo_total_benchmark_s:.1f} segundos",
)


[01/24] circuito_horario, ruído 0.50: concluído em 25.3 s
[02/24] circuito_horario, ruído 0.75: concluído em 50.1 s
[03/24] circuito_horario, ruído 1.00: concluído em 74.6 s
[04/24] circuito_horario, ruído 1.25: concluído em 99.2 s
[05/24] circuito_horario, ruído 1.50: concluído em 123.7 s
[06/24] circuito_horario, ruído 2.00: concluído em 148.3 s
[07/24] circuito_antihorario, ruído 0.50: concluído em 172.8 s
[08/24] circuito_antihorario, ruído 0.75: concluído em 197.5 s
[09/24] circuito_antihorario, ruído 1.00: concluído em 222.2 s
[10/24] circuito_antihorario, ruído 1.25: concluído em 246.6 s
[11/24] circuito_antihorario, ruído 1.50: concluído em 270.7 s
[12/24] circuito_antihorario, ruído 2.00: concluído em 294.7 s
[13/24] troca_central_longa, ruído 0.50: concluído em 318.5 s
[14/24] troca_central_longa, ruído 0.75: concluído em 342.6 s
[15/24] troca_central_longa, ruído 1.00: concluído em 366.6 s
[16/24] troca_central_longa, ruído 1.25: concluído em 390.4 s
[17/24] troca_central_lo

## 6. Resumo e gravação dos resultados

A mediana representa a latência típica. O P95 indica um limite abaixo do qual ficaram 95% das chamadas observadas. O tempo por observação é apenas a latência da sequência dividida por seis.


In [7]:
def resumir_latencias(dados, grupos):
    resumo = (
        dados.groupby(grupos, as_index=False)["tempo_ns"]
        .agg(
            n="size",
            media_ns="mean",
            desvio_padrao_ns="std",
            minimo_ns="min",
            q1_ns=lambda serie: serie.quantile(0.25),
            mediana_ns="median",
            q3_ns=lambda serie: serie.quantile(0.75),
            p95_ns=lambda serie: serie.quantile(0.95),
            maximo_ns="max",
        )
    )
    resumo["media_ms"] = resumo["media_ns"] / 1e6
    resumo["mediana_ms"] = resumo["mediana_ns"] / 1e6
    resumo["q1_ms"] = resumo["q1_ns"] / 1e6
    resumo["q3_ms"] = resumo["q3_ns"] / 1e6
    resumo["p95_ms"] = resumo["p95_ns"] / 1e6
    return resumo


resumo_geral = resumir_latencias(
    latencias,
    ["metodo", "quantidade_observacoes"],
)
resumo_geral["mediana_ms_por_observacao"] = (
    resumo_geral["mediana_ms"]
    / resumo_geral["quantidade_observacoes"]
)
resumo_geral["p95_ms_por_observacao"] = (
    resumo_geral["p95_ms"]
    / resumo_geral["quantidade_observacoes"]
)

resumo_por_condicao = resumir_latencias(
    latencias,
    [
        "metodo",
        "nome_sequencia",
        "fator_ruido",
        "quantidade_observacoes",
    ],
)

ordem_exibicao = list(pipelines)
resumo_geral["metodo"] = pd.Categorical(
    resumo_geral["metodo"],
    categories=ordem_exibicao,
    ordered=True,
)
resumo_geral = resumo_geral.sort_values("metodo")
resumo_geral["metodo"] = (
    resumo_geral["metodo"].astype(str)
)

display(
    resumo_geral[
        [
            "metodo",
            "n",
            "mediana_ms",
            "p95_ms",
            "media_ms",
            "mediana_ms_por_observacao",
        ]
    ].round(6)
)

PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

caminho_latencias = (
    PASTA_RESULTADOS / "latencias_metodos_teste.csv.gz"
)
caminho_resumo = (
    PASTA_RESULTADOS / "resumo_latencias_metodos_teste.csv"
)
caminho_condicoes = (
    PASTA_RESULTADOS
    / "resumo_latencias_por_condicao_teste.csv"
)
caminho_ambiente = (
    PASTA_RESULTADOS / "ambiente_desempenho.json"
)

caminho_temporario = caminho_latencias.with_name(
    caminho_latencias.name + ".tmp"
)
latencias.to_csv(
    caminho_temporario,
    index=False,
    compression="gzip",
)
caminho_temporario.replace(caminho_latencias)

resumo_geral.to_csv(caminho_resumo, index=False)
resumo_por_condicao.to_csv(
    caminho_condicoes,
    index=False,
)
caminho_ambiente.write_text(
    json.dumps(
        ambiente,
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print("Arquivos gravados:")
for caminho in [
    caminho_latencias,
    caminho_resumo,
    caminho_condicoes,
    caminho_ambiente,
]:
    print("-", caminho.relative_to(RAIZ))


,metodo,n,mediana_ms,p95_ms,media_ms,mediana_ms_por_observacao
0,espacial,24000,2.002583,2.313850,2.038815,0.333764
2,forward,24000,5.080644,5.521725,5.107250,0.846774
4,viterbi,24000,4.816742,5.239838,4.844152,0.802790
1,espacial_com_abstencao,24000,4.818374,5.231017,4.842359,0.803062
3,forward_com_abstencao,24000,6.184176,6.648378,6.203389,1.030696


Arquivos gravados:
- resultados/latencias_metodos_teste.csv.gz
- resultados/resumo_latencias_metodos_teste.csv
- resultados/resumo_latencias_por_condicao_teste.csv
- resultados/ambiente_desempenho.json


## 7. Manifesto e verificações finais

Os hashes ligam os arquivos de desempenho aos bytes efetivamente produzidos. O manifesto também registra as decisões metodológicas necessárias para interpretar os tempos corretamente.


In [8]:
def sha256(caminho):
    return hashlib.sha256(caminho.read_bytes()).hexdigest()


artefatos = [
    caminho_latencias,
    caminho_resumo,
    caminho_condicoes,
    caminho_ambiente,
]

manifesto_desempenho = {
    "versao_esquema": 1,
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
    "implementacao": {
        "notebook_base": str(
            NOTEBOOK_BASE.relative_to(RAIZ)
        ),
        "commit_base": commit_base,
        "sha256_fontes_notebook_base": (
            ambiente["sha256_fontes_notebook_base"]
        ),
    },
    "configuracao": {
        "rotas": rotas_teste,
        "fatores_ruido": fatores_ruido,
        "sementes": {
            "inicio": min(sementes_teste),
            "fim_inclusivo": max(sementes_teste),
            "quantidade": len(sementes_teste),
        },
        "quantidade_sequencias": quantidade_sequencias,
        "quantidade_observacoes": int(
            quantidade_sequencias
            * len(sequencias_inspecao[rotas_teste[0]])
        ),
        "quantidade_medicoes": len(latencias),
        "escala_transicao_m": escala_transicao,
        "limiares": manifesto_teste["limiares"],
        "semente_ordem_metodos": 20260908,
    },
    "medicao": {
        "relogio": "time.perf_counter_ns",
        "aquecimentos_por_pipeline": 10,
        "ordem_metodos_aleatorizada": True,
        "gc_desativado_durante_medicao": True,
        "simulacao_incluida": False,
        "matriz_transicao_incluida": False,
        "leitura_escrita_arquivos_incluida": False,
        "unidade_cronometrada": (
            "sequencia completa de seis observacoes"
        ),
        "viterbi_com_abstencao": False,
    },
    "ambiente": ambiente,
    "tempo_total_benchmark_s": tempo_total_benchmark_s,
    "sha256_artefatos": {
        str(caminho.relative_to(RAIZ)): sha256(caminho)
        for caminho in artefatos
    },
}

caminho_manifesto = (
    PASTA_RESULTADOS / "manifesto_desempenho.json"
)
caminho_manifesto.write_text(
    json.dumps(
        manifesto_desempenho,
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

invalidos = []

for nome, hash_esperado in manifesto_desempenho[
    "sha256_artefatos"
].items():
    caminho = RAIZ / nome

    if not caminho.is_file():
        invalidos.append(f"{nome}: ausente")
    elif sha256(caminho) != hash_esperado:
        invalidos.append(f"{nome}: hash divergente")

print("Quantidade de métodos:", latencias["metodo"].nunique())
print("Quantidade de condições:", quantidade_condicoes)
print("Quantidade de sequências:", quantidade_sequencias)
print("Quantidade de medições:", len(latencias))
print("Tempos positivos:", bool((latencias["tempo_ns"] > 0).all()))
print("Artefatos inválidos:", invalidos)
print("Manifesto válido:", not invalidos)
print("Manifesto:", caminho_manifesto.relative_to(RAIZ))


Quantidade de métodos: 5
Quantidade de condições: 24
Quantidade de sequências: 24000
Quantidade de medições: 120000
Tempos positivos: True
Artefatos inválidos: []
Manifesto válido: True
Manifesto: resultados/manifesto_desempenho.json


## Como interpretar sem extrapolar

- A mediana descreve uma chamada típica; o P95 descreve a cauda observada no computador do experimento.
- Comparações entre métodos valem para estas implementações em Python e para seis estados e seis observações por sequência.
- Esses números não demonstram, por si só, desempenho em tempo real no Meta Quest 3 nem no sistema completo de percepção, localização e renderização.
- A simulação e a análise estatística anteriores permanecem inalteradas. Este notebook acrescenta somente a dimensão de custo computacional prevista na etapa 12.
